# Progetto 2: Livelli di Isolamento in PostgreSQL
## Comportamento di PostgreSQL nella gestione dei diversi livelli di isolamento

Questo progetto studia e sperimenta il comportamento di PostgreSQL nella gestione dei livelli di isolamento, con particolare riferimento alle principali anomalie che possono verificarsi durante l'esecuzione concorrente di transazioni.

### Livelli di isolamento esaminati:
1. **READ COMMITTED** - Comportamento predefinito in PostgreSQL
2. **REPEATABLE READ** - Fornisce una maggiore consistenza
3. **SERIALIZABLE** - Il livello più restrittivo che garantisce massima consistenza

### Obiettivi dei test:
1. Inserimento con cicli di lettura/scrittura
2. Modifica concorrenziale dello stesso campo
3. Inserimento di un campo chiave primaria già esistente

In [1]:
# Importazione librerie necessarie
from faker import Faker
import random
import psycopg2
import time

## Connessione al database PostgreSQL

In [2]:
# Connessione del primo client al database
conn1 = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="postgres",
    host="localhost",
    port="5432"
)

# Connessione del secondo client al database
conn2 = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="postgres",
    host="localhost",
    port="5432"
)

# Cursore per interrogazioni per il primo client
cursor1 = conn1.cursor()

# Cursore per interrogazioni per il secondo client
cursor2 = conn2.cursor()

print("Connessione ai client stabilita con successo")

Connessione ai client stabilita con successo


## Creazione e popolamento della tabella Studente

In [3]:
# Creo e popolo la tabella Studente
cursor1.execute("""
    DROP TABLE IF EXISTS Studente CASCADE;
    
    CREATE TABLE Studente (
        matricola VARCHAR(10) PRIMARY KEY,
        nome VARCHAR(100) NOT NULL,
        cognome VARCHAR(100) NOT NULL,
        cfu INTEGER NOT NULL
    );
""")

conn1.commit()

# Generatore di dati casuali
fake = Faker('it_IT')

# Funzione per generare dati casuali per gli studenti
def generate_students(num_students):
    students = []
    for i in range(1, num_students + 1):
        matricola = f"{10000 + i}"
        nome = fake.first_name()
        cognome = fake.last_name()
        cfu = random.randint(0, 180)
        students.append((matricola, nome, cognome, cfu))
    return students

# Inserisco 50 studenti casuali
students = generate_students(50)
cursor1.executemany(
    "INSERT INTO Studente (matricola, nome, cognome, cfu) VALUES (%s, %s, %s, %s)",
    students
)

conn1.commit()
print("Tabella Studente creata e popolata con 50 studenti")

Tabella Studente creata e popolata con 50 studenti


## Test 1: Inserimento con cicli di lettura/scrittura

Il test verifica il comportamento quando due transazioni leggono contemporaneamente un valore (il massimo CFU presente) e lo utilizzano per un nuovo inserimento.

In [4]:
def test_read_write_cycle(isolation_level):
    print(f"\n=== Test con livello di isolamento {isolation_level} ===")
    
    # Impostazione del livello di isolamento e avvio transazione per client 1
    cursor1.execute(f"START TRANSACTION ISOLATION LEVEL {isolation_level};")
    
    # Client 1 legge il massimo CFU esistente
    cursor1.execute("SELECT MAX(cfu) FROM Studente;")
    max_cfu1 = cursor1.fetchone()[0]
    print(f"Client 1: Massimo CFU rilevato = {max_cfu1}")
    
    # Impostazione del livello di isolamento e avvio transazione per client 2
    cursor2.execute(f"START TRANSACTION ISOLATION LEVEL {isolation_level};")
    
    # Client 2 legge il massimo CFU esistente
    cursor2.execute("SELECT MAX(cfu) FROM Studente;")
    max_cfu2 = cursor2.fetchone()[0]
    print(f"Client 2: Massimo CFU rilevato = {max_cfu2}")
    
    try:
        # Client 1 inserisce un nuovo studente con CFU pari al massimo + 10
        new_cfu1 = max_cfu1 + 10
        cursor1.execute(f"""
            INSERT INTO Studente (matricola, nome, cognome, cfu) 
            VALUES ('99001', 'Mario', 'Rossi', {new_cfu1});
        """)
        print(f"Client 1: Inserito studente con {new_cfu1} CFU")
        
        # Client 2 inserisce un nuovo studente con CFU pari al massimo + 5
        new_cfu2 = max_cfu2 + 5
        cursor2.execute(f"""
            INSERT INTO Studente (matricola, nome, cognome, cfu) 
            VALUES ('99002', 'Laura', 'Bianchi', {new_cfu2});
        """)
        print(f"Client 2: Inserito studente con {new_cfu2} CFU")
        
        # Commit delle transazioni
        conn1.commit()
        print("Client 1: Commit eseguito con successo")
        
        conn2.commit()
        print("Client 2: Commit eseguito con successo")
        
    except Exception as e:
        print(f"Errore: {e}")
        conn1.rollback()
        conn2.rollback()
    
    # Verifica finale
    cursor1.execute("SELECT matricola, cfu FROM Studente WHERE matricola IN ('99001', '99002') ORDER BY cfu DESC;")
    result = cursor1.fetchall()
    print("\nRisultato finale:")
    for row in result:
        print(f"Matricola: {row[0]}, CFU: {row[1]}")

# Eseguo il test con diversi livelli di isolamento
print("TEST 1: INSERIMENTO CON CICLI DI LETTURA/SCRITTURA")

# Pulizia preliminare
cursor1.execute("DELETE FROM Studente WHERE matricola IN ('99001', '99002');")
conn1.commit()

# Test con READ COMMITTED
test_read_write_cycle("READ COMMITTED")

# Pulizia tra i test
cursor1.execute("DELETE FROM Studente WHERE matricola IN ('99001', '99002');")
conn1.commit()

# Test con REPEATABLE READ
test_read_write_cycle("REPEATABLE READ")

# Pulizia tra i test
cursor1.execute("DELETE FROM Studente WHERE matricola IN ('99001', '99002');")
conn1.commit()

# Test con SERIALIZABLE
test_read_write_cycle("SERIALIZABLE")

TEST 1: INSERIMENTO CON CICLI DI LETTURA/SCRITTURA

=== Test con livello di isolamento READ COMMITTED ===
Client 1: Massimo CFU rilevato = 168
Client 2: Massimo CFU rilevato = 168
Client 1: Inserito studente con 178 CFU
Client 2: Inserito studente con 173 CFU
Client 1: Commit eseguito con successo
Client 2: Commit eseguito con successo

Risultato finale:
Matricola: 99001, CFU: 178
Matricola: 99002, CFU: 173

=== Test con livello di isolamento REPEATABLE READ ===
Client 1: Massimo CFU rilevato = 168
Client 2: Massimo CFU rilevato = 168
Client 1: Inserito studente con 178 CFU
Client 2: Inserito studente con 173 CFU
Client 1: Commit eseguito con successo
Client 2: Commit eseguito con successo

Risultato finale:
Matricola: 99001, CFU: 178
Matricola: 99002, CFU: 173

=== Test con livello di isolamento SERIALIZABLE ===
Client 1: Massimo CFU rilevato = 168
Client 2: Massimo CFU rilevato = 168
Client 1: Inserito studente con 178 CFU
Client 2: Inserito studente con 173 CFU
Client 1: Commit eseg

## Test 2: Modifica concorrenziale dello stesso campo

Il test verifica il comportamento quando due transazioni tentano di modificare contemporaneamente lo stesso campo dello stesso record.

In [5]:
def test_concurrent_update(isolation_level):
    print(f"\n=== Test con livello di isolamento {isolation_level} ===")
    
    # Preparo il record di test
    cursor1.execute("DELETE FROM Studente WHERE matricola = '10001';")
    cursor1.execute("INSERT INTO Studente (matricola, nome, cognome, cfu) VALUES ('10001', 'Test', 'User', 60);")
    conn1.commit()
    
    # Impostazione del livello di isolamento e avvio transazione per client 1
    cursor1.execute(f"START TRANSACTION ISOLATION LEVEL {isolation_level};")
    
    # Imposto un timeout di 3 secondi per evitare blocchi infiniti
    cursor2.execute("SET statement_timeout = 3000;")  # 3000 ms = 3 secondi
    
    # Impostazione del livello di isolamento e avvio transazione per client 2
    cursor2.execute(f"START TRANSACTION ISOLATION LEVEL {isolation_level};")
    
    try:
        # Client 1 legge il record
        cursor1.execute("SELECT cfu FROM Studente WHERE matricola = '10001';")
        cfu1 = cursor1.fetchone()[0]
        print(f"Client 1: Letto CFU = {cfu1}")
        
        # Client 2 legge il record
        cursor2.execute("SELECT cfu FROM Studente WHERE matricola = '10001';")
        cfu2 = cursor2.fetchone()[0]
        print(f"Client 2: Letto CFU = {cfu2}")
        
        # Client 1 aggiorna il record
        cursor1.execute("UPDATE Studente SET cfu = cfu + 10 WHERE matricola = '10001';")
        print("Client 1: Aggiornati CFU = cfu + 10")
        
        # Client 2 aggiorna il record - Questo potrebbe bloccarsi
        try:
            cursor2.execute("UPDATE Studente SET cfu = cfu + 5 WHERE matricola = '10001';")
            print("Client 2: Aggiornati CFU = cfu + 5")
        except psycopg2.errors.QueryCanceled as e:
            print("Client 2: Timeout raggiunto durante l'attesa del lock")
            print("Questo è un comportamento normale in PostgreSQL quando più client cercano di modificare lo stesso record")
            raise e
        
        # Commit delle transazioni
        conn1.commit()
        print("Client 1: Commit eseguito con successo")
        
        conn2.commit()
        print("Client 2: Commit eseguito con successo")
        
    except Exception as e:
        print(f"Errore: {e}")
        # Assicuriamoci di fare rollback di entrambe le connessioni
        try:
            conn1.rollback()
            print("Client 1: Rollback eseguito")
        except:
            pass
        try:
            conn2.rollback()
            print("Client 2: Rollback eseguito")
        except:
            pass
    finally:
        # Reset del timeout alla fine del test
        cursor2.execute("SET statement_timeout = 0;")  # Torniamo al valore predefinito (nessun timeout)
    
    # Verifica finale
    cursor1.execute("SELECT matricola, cfu FROM Studente WHERE matricola = '10001';")
    result = cursor1.fetchone()
    if result:
        print(f"\nRisultato finale: Matricola {result[0]}, CFU: {result[1]}")
    else:
        print("\nRisultato finale: Record non trovato")

# Eseguo il test con diversi livelli di isolamento
print("\nTEST 2: MODIFICA CONCORRENZIALE DELLO STESSO CAMPO")

# Test con READ COMMITTED
test_concurrent_update("READ COMMITTED")

# Test con REPEATABLE READ
test_concurrent_update("REPEATABLE READ")

# Test con SERIALIZABLE
test_concurrent_update("SERIALIZABLE")


TEST 2: MODIFICA CONCORRENZIALE DELLO STESSO CAMPO

=== Test con livello di isolamento READ COMMITTED ===
Client 1: Letto CFU = 60
Client 2: Letto CFU = 60
Client 1: Aggiornati CFU = cfu + 10
Client 2: Timeout raggiunto durante l'attesa del lock
Questo è un comportamento normale in PostgreSQL quando più client cercano di modificare lo stesso record
Errore: ERRORE:  annullamento dell'istruzione a causa di timeout
CONTEXT:  durante la modifica della tupla (0,57) nella relazione "studente"

Client 1: Rollback eseguito
Client 2: Rollback eseguito

Risultato finale: Matricola 10001, CFU: 60

=== Test con livello di isolamento REPEATABLE READ ===
Client 1: Letto CFU = 60
Client 2: Letto CFU = 60
Client 1: Aggiornati CFU = cfu + 10
Client 2: Timeout raggiunto durante l'attesa del lock
Questo è un comportamento normale in PostgreSQL quando più client cercano di modificare lo stesso record
Errore: ERRORE:  annullamento dell'istruzione a causa di timeout
CONTEXT:  durante la modifica della tupl

## Test 3: Inserimento di un campo chiave primaria già esistente

Il test verifica il comportamento quando due transazioni tentano di inserire contemporaneamente un record con la stessa chiave primaria.

In [6]:
def test_primary_key_conflict(isolation_level):
    print(f"\n=== Test con livello di isolamento {isolation_level} ===")
    
    # Rimuovo eventuali record di test precedenti
    cursor1.execute("DELETE FROM Studente WHERE matricola = '20001';")
    conn1.commit()
    
    # Impostazione del livello di isolamento e avvio transazione per client 1
    cursor1.execute(f"START TRANSACTION ISOLATION LEVEL {isolation_level};")
    
    # Imposto un timeout di 3 secondi per evitare blocchi infiniti
    cursor2.execute("SET statement_timeout = 3000;")  # 3000 ms = 3 secondi
    
    # Impostazione del livello di isolamento e avvio transazione per client 2
    cursor2.execute(f"START TRANSACTION ISOLATION LEVEL {isolation_level};")
    
    try:
        # Client 1 verifica se la matricola esiste
        cursor1.execute("SELECT COUNT(*) FROM Studente WHERE matricola = '20001';")
        count1 = cursor1.fetchone()[0]
        print(f"Client 1: La matricola 20001 {'esiste già' if count1 > 0 else 'non esiste'}")
        
        # Client 2 verifica se la matricola esiste
        cursor2.execute("SELECT COUNT(*) FROM Studente WHERE matricola = '20001';")
        count2 = cursor2.fetchone()[0]
        print(f"Client 2: La matricola 20001 {'esiste già' if count2 > 0 else 'non esiste'}")
        
        # Client 1 inserisce un nuovo studente con la matricola 20001
        cursor1.execute("""
            INSERT INTO Studente (matricola, nome, cognome, cfu) 
            VALUES ('20001', 'Franco', 'Neri', 100);
        """)
        print("Client 1: Inserito studente con matricola 20001")
        
        # Client 2 inserisce un nuovo studente con la stessa matricola 20001
        try:
            cursor2.execute("""
                INSERT INTO Studente (matricola, nome, cognome, cfu) 
                VALUES ('20001', 'Anna', 'Verdi', 120);
            """)
            print("Client 2: Inserito studente con matricola 20001")
        except psycopg2.errors.QueryCanceled as e:
            print("Client 2: Timeout raggiunto durante l'attesa del lock")
            print("Questo è un comportamento normale quando più client cercano di inserire lo stesso valore chiave")
            raise e
        except psycopg2.errors.UniqueViolation as e:
            print("Client 2: Violazione del vincolo di chiave primaria")
            print("Questo errore si verifica quando il primo client ha già committato l'inserimento")
            raise e
        
        # Commit delle transazioni
        conn1.commit()
        print("Client 1: Commit eseguito con successo")
        
        conn2.commit()
        print("Client 2: Commit eseguito con successo")
        
    except Exception as e:
        print(f"Errore: {e}")
        # Assicuriamoci di fare rollback di entrambe le connessioni
        try:
            conn1.rollback()
            print("Client 1: Rollback eseguito")
        except:
            pass
        try:
            conn2.rollback()
            print("Client 2: Rollback eseguito")
        except:
            pass
    finally:
        # Reset del timeout alla fine del test
        cursor2.execute("SET statement_timeout = 0;")  # Torniamo al valore predefinito (nessun timeout)
    
    # Verifica finale
    cursor1.execute("SELECT matricola, nome, cognome, cfu FROM Studente WHERE matricola = '20001';")
    result = cursor1.fetchall()
    print("\nRisultato finale:")
    if result:
        for row in result:
            print(f"Matricola: {row[0]}, Nome: {row[1]}, Cognome: {row[2]}, CFU: {row[3]}")
    else:
        print("Nessun record trovato con matricola 20001")

# Eseguo il test con diversi livelli di isolamento
print("\nTEST 3: INSERIMENTO CON CHIAVE PRIMARIA GIÀ ESISTENTE")

# Test con READ COMMITTED
test_primary_key_conflict("READ COMMITTED")

# Test con REPEATABLE READ
test_primary_key_conflict("REPEATABLE READ")

# Test con SERIALIZABLE
test_primary_key_conflict("SERIALIZABLE")


TEST 3: INSERIMENTO CON CHIAVE PRIMARIA GIÀ ESISTENTE

=== Test con livello di isolamento READ COMMITTED ===
Client 1: La matricola 20001 non esiste
Client 2: La matricola 20001 non esiste
Client 1: Inserito studente con matricola 20001
Client 2: Timeout raggiunto durante l'attesa del lock
Questo è un comportamento normale quando più client cercano di inserire lo stesso valore chiave
Errore: ERRORE:  annullamento dell'istruzione a causa di timeout
CONTEXT:  durante l'inserimento della tupla di indice (0,64) nella relazione "studente_pkey"

Client 1: Rollback eseguito
Client 2: Rollback eseguito

Risultato finale:
Nessun record trovato con matricola 20001

=== Test con livello di isolamento REPEATABLE READ ===
Client 1: La matricola 20001 non esiste
Client 2: La matricola 20001 non esiste
Client 1: Inserito studente con matricola 20001
Client 2: Timeout raggiunto durante l'attesa del lock
Questo è un comportamento normale quando più client cercano di inserire lo stesso valore chiave
Er

## Conclusioni

Dai test effettuati, possiamo osservare il diverso comportamento di PostgreSQL nei vari livelli di isolamento:

1. **READ COMMITTED**:
   - Permette di vedere solo i dati committati
   - Non rileva anomalie di concorrenza come write skew
   - Può causare problemi di consistenza in operazioni complesse

2. **REPEATABLE READ**:
   - Garantisce che le letture ripetute dello stesso dato restituiscano lo stesso valore
   - Usa il controllo di concorrenza multiversione (MVCC)
   - Rileva alcuni tipi di anomalie, ma non tutte

3. **SERIALIZABLE**:
   - Il livello più restrittivo e sicuro
   - Simula un'esecuzione sequenziale delle transazioni
   - Rileva e previene tutte le anomalie di concorrenza
   - Può causare errori di serializzazione che richiedono il riavvio delle transazioni

PostgreSQL dimostra una buona gestione dei conflitti grazie al suo sistema MVCC, che permette di lavorare con diverse versioni dei dati contemporaneamente senza blocchi eccessivi. La scelta del livello di isolamento dipende dalle esigenze specifiche dell'applicazione, bilanciando consistenza e performance.